### 02 - Pandas

**Documentação Pandas**
    - **https://pandas.pydata.org/docs/** 

**Objetivo**
    - Entender como **Series** e **DataFrame** funcionam internamente e como isso impacta performance, memória e corretude em pipelines de dados.

### Roadmap
1. Arquitetura lógica do pandas
2. Índice, alinhamento e semântica de rótulos
3. Dtypes e memória
4. **groupby**, **merge**, missing values
5. Criação de datasets sintéticos simples para prática


In [109]:
import numpy as np
import pandas as pd
import seaborn as sns

np.random.seed(42)
sns.set_theme(style='whitegrid', context='notebook')

### 1 - Conceito: Series e DataFrame

- **Series**: vetor 1D com índice explícito
- **DataFrame**: coleção de **Series** alinhadas por índice

Pandas privilegia semântica de rótulos (labels), não apenas posição.

In [110]:
s = pd.Series([10, 20, 30], index=['a', 'b', 'c'])
df = pd.DataFrame({
    'produto': ['A', 'B', 'C'],
    'preco': [12.5, 18.0, 7.9],
    'estoque': [10, 6, 25],
})
print(s['a'])
df

10


,produto,preco,estoque
0,A,12.5,10
1,B,18.0,6
2,C,7.9,25


### 2 - Alinhamento por índice

Operações entre objetos pandas fazem união/interseção de índices antes de operar. Isso aumenta segurança semântica, mas pode custar performance e introduzir **NaN** inesperado.

In [111]:
s1 = pd.Series([1, 2, 3], index=['x', 'y', 'z'])
s2 = pd.Series([10, 20, 30], index=['y', 'z', 'w'])

print('s1 + s2:')
print(s1 + s2)


s1 + s2:
w     NaN
x     NaN
y    12.0
z    23.0
dtype: float64


### 3 - Dtypes, memória e categorias

Escolher dtype correto reduz memória e acelera operações.
- **object** é flexível, porém custoso
- **category** representa rótulos com códigos inteiros + dicionário

In [112]:
n = 200_000
df_raw = pd.DataFrame({
    'uf_obj': np.random.choice(['SP', 'RJ', 'MG', 'BA', 'RS'], size=n),
    'valor': np.random.rand(n),
})

mem_obj = df_raw.memory_usage(deep=True).sum()
df_clean = df_raw.copy()
df_clean['uf_obj'] = df_clean['uf_obj'].astype('category')
mem_cat = df_clean.memory_usage(deep=True).sum()

print(f'Memória com object: {mem_obj/1e6:.2f} MB')
print(f'Memória com category: {mem_cat/1e6:.2f} MB')
print(f'Redução: {(1 - mem_cat/mem_obj)*100:.2f}%')


Memória com object: 13.40 MB
Memória com category: 1.80 MB
Redução: 86.56%


### 4 - **groupby** internamente

**groupby** executa a estratégia split-apply-combine:
1. fatorização das chaves em códigos
2. agregação por bloco/código
3. recomposição do índice de saída

Operações vetorizadas em agregações nativas são muito mais rápidas que **apply** Python.

In [113]:
ag = (
    df_clean
    .groupby('uf_obj', observed=True)
    .agg(media_valor=('valor', 'mean'), soma_valor=('valor', 'sum'), qtd=('valor', 'size'))
    .sort_values('media_valor', ascending=False)
)
ag


,media_valor,soma_valor,qtd
uf_obj,,,
MG,0.501307,19855.251525,39607
RJ,0.500362,20163.595449,40298
SP,0.499974,19935.466935,39873
RS,0.499417,20101.519932,40250
BA,0.497932,19903.321162,39972


### 5 - **merge**/**join**: semântica relacional

**merge** aplica conceitos de banco relacional com possíveis estratégias internas de hash-join e sort-merge dependendo do caso.

Pontos críticos:
- cardinalidade (1:1, 1:N, N:N)
- chaves duplicadas
- **how** (inner, left, right, outer)


In [114]:
left = pd.DataFrame({'id': [1, 2, 3, 3], 'x': [10, 20, 30, 31]})
right = pd.DataFrame({'id': [2, 3, 4], 'y': ['a', 'b', 'c']})

print('INNER')
print(pd.merge(left, right, on='id', how='inner'))

print('\nLEFT')
print(pd.merge(left, right, on='id', how='left'))


INNER
   id   x  y
0   2  20  a
1   3  30  b
2   3  31  b

LEFT
   id   x    y
0   1  10  NaN
1   2  20    a
2   3  30    b
3   3  31    b


### 6 - Missing values e tipos anuláveis

**NaN** em float não é igual a **None** semântico em colunas inteiras/booleanas.
Use dtypes anuláveis (**Int64**, **boolean**, **string**) para evitar coerções indesejadas.

In [115]:
df_miss = pd.DataFrame({'a': [1, None, 3], 'b': [True, None, False]})
print(df_miss.dtypes)

# Versão com tipos anuláveis

df_miss2 = pd.DataFrame({
    'a': pd.Series([1, None, 3], dtype='Int64'),
    'b': pd.Series([True, None, False], dtype='boolean')
})
print('\nTipos anuláveis:')
print(df_miss2.dtypes)


a    float64
b     object
dtype: object

Tipos anuláveis:
a      Int64
b    boolean
dtype: object


### 7 - Dados sintéticos

Nesta seção, usamos apenas dados forjados para praticar pandas sem depender de arquivos externos.

**Exemplo 1: vendas mensais**
- poucas colunas
- tipos fáceis de inspecionar

**Exemplo 2: cadastro de instituições**
- mistura de variáveis categóricas e numéricas
- volume suficiente para testes simples


In [121]:
inicio = pd.Timestamp('2020-01-01')
fim = pd.Timestamp.today().normalize()
datas = pd.date_range(start=inicio, end=fim, freq='MS')

df_raw = pd.DataFrame({
    'mes': datas,
    'categoria': np.random.choice(['A', 'B', 'C'], size=len(datas), p=[0.5, 0.3, 0.2]),
    'vendas': np.random.randint(80, 220, size=len(datas)),
    'ticket_medio': np.random.normal(45, 8, size=len(datas)).round(2)
})

df_clean = df_raw.copy()
df_clean['ticket_medio'] = df_clean['ticket_medio'].clip(lower=20)
df_clean = df_clean[df_clean['mes'].between(inicio, fim)].copy()


In [122]:
df_clean.head(100)

,mes,categoria,vendas,ticket_medio
0,2020-01-01,A,211,41.17
1,2020-02-01,B,152,29.35
2,2020-03-01,C,217,46.13
3,2020-04-01,A,91,30.76
4,2020-05-01,C,139,46.91
...,...,...,...,...
69,2025-10-01,A,122,35.21
70,2025-11-01,B,141,41.90
71,2025-12-01,C,181,41.99
72,2026-01-01,C,104,58.38


In [118]:
df_clean.dtypes

mes             datetime64[us]
categoria                  str
vendas                   int64
ticket_medio           float64
dtype: object

In [119]:
n = 500

df_raw = pd.DataFrame({
    'id': np.arange(1, n + 1),
    'regiao': np.random.choice(['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul'], size=n),
    'rede': np.random.choice(['Publica', 'Privada'], size=n, p=[0.25, 0.75]),
    'docentes': np.random.randint(20, 260, size=n),
    'alunos': np.random.randint(200, 8000, size=n),
    'nota': np.random.normal(3.4, 0.7, size=n).clip(1, 5).round(2)
})

df_clean = df_raw.copy()
print('Shape: ', df_clean.shape)
print('\nTipos:\n', df_clean.dtypes)


Shape:  (500, 6)

Tipos:
 id            int64
regiao          str
rede            str
docentes      int64
alunos        int64
nota        float64
dtype: object


In [120]:
df_clean.head()

,id,regiao,rede,docentes,alunos,nota
0,1,Sul,Privada,124,1115,2.79
1,2,Sul,Publica,161,4289,1.74
2,3,Sul,Privada,65,4221,3.74
3,4,Norte,Privada,72,580,3.23
4,5,Centro-Oeste,Privada,90,7678,4.12


### 8 - Erros Comuns

- Esquecer encoding e separador corretos
- Ler coluna categórica como **object** gigante
- Usar **apply** linha a linha sem necessidade
- Fazer **merge** sem validar cardinalidade

**Checklist**
1. **df.info()** e **memory_usage(deep=True)**
2. validar chaves antes de **merge**
3. padronizar tipos no início do pipeline
